In [0]:
# Load data into dbx.default schema
spark.sql("""
    CREATE TABLE IF NOT EXISTS dbx.default.forecast_daily_calendar_imperial AS
    SELECT * FROM samples.accuweather.forecast_daily_calendar_imperial
""")

# Prepare data for regression (example: predicting 'temperature' based on 'day_of_year')
df = spark.sql("SELECT dayofyear(date) AS day_of_year, temperature_avg FROM dbx.default.forecast_daily_calendar_imperial WHERE temperature_avg IS NOT NULL")
from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(inputCols=["day_of_year"], outputCol="features")
df_features = assembler.transform(df)

# Train regression model
from pyspark.ml.regression import LinearRegression
lr = LinearRegression(featuresCol="features", labelCol="temperature_avg")
model = lr.fit(df_features)

# Predict future weather (example: for future days)
future_days = spark.createDataFrame([(366,), (367,), (368,)], ["day_of_year"])
future_features = assembler.transform(future_days)
predictions = model.transform(future_features)
display(predictions)

day_of_year,features,prediction
366,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""366.0""]}",70.22238261931847
367,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""367.0""]}",70.18176619074868
368,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""368.0""]}",70.1411497621789


In [0]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Convert Spark DataFrame to pandas and prepare features
pdf = df.toPandas()
X = pdf[["day_of_year"]]
y = pdf["temperature_avg"]

# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Custom sklearn pipeline: scaler + Ridge regression (preprocessing encapsulated)
sk_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge",  Ridge(alpha=1.0))
])
sk_pipeline.fit(X_train, y_train)

# Evaluation metrics
y_pred    = sk_pipeline.predict(X_test)
rmse      = np.sqrt(mean_squared_error(y_test, y_pred))
mae       = mean_absolute_error(y_test, y_pred)
r2        = r2_score(y_test, y_pred)
residuals = y_test.values - y_pred

metrics_df = pd.DataFrame({
    "Metric": ["RMSE", "MAE", "R²", "Residual Mean", "Residual Std"],
    "Value":  [
        round(rmse, 4),
        round(mae, 4),
        round(r2, 4),
        round(float(np.mean(residuals)), 4),
        round(float(np.std(residuals)), 4),
    ]
})
display(metrics_df)

Metric,Value
RMSE,12.9119
MAE,10.192
R²,-0.0049
Residual Mean,-0.5686
Residual Std,12.8993


In [0]:
import mlflow
from mlflow.models import infer_signature
import pandas as pd

# ── Custom PyFunc model wrapping the sklearn pipeline ──────────────────────────
class TemperatureForecaster(mlflow.pyfunc.PythonModel):
    """Predicts daily average temperature from day-of-year."""

    def __init__(self, model):
        self._model = model

    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        preds = self._model.predict(model_input[["day_of_year"]])
        return pd.DataFrame({"temperature_avg_prediction": preds})

# ── MLflow: log + register to Unity Catalog ───────────────────────────────────
mlflow.set_registry_uri("databricks-uc")
registered_model_name = "dbx.default.temperature_forecaster"

signature = infer_signature(
    X_train,
    pd.DataFrame({"temperature_avg_prediction": sk_pipeline.predict(X_train)})
)

with mlflow.start_run(run_name="temperature_forecast_custom"):
    mlflow.log_params({"model_type": "Ridge", "alpha": 1.0, "test_size": 0.2})
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    model_info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=TemperatureForecaster(sk_pipeline),
        signature=signature,
        input_example=X_train.head(3),
        registered_model_name=registered_model_name,
    )

print(f"Model registered : {registered_model_name}")
print(f"Model version    : {model_info.registered_model_version}")

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/07/26 04:49:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://adb-7405609165985958.18.azuredatabricks.net/ml/experiments/4233700580339565/models/m-3e6e9b39f76d4d7f

Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

🔗 Created version '1' of model 'dbx.default.temperature_forecaster': https://adb-7405609165985958.18.azuredatabricks.net/explore/data/models/dbx/default/temperature_forecaster/version/1?o=7405609165985958


Model registered : dbx.default.temperature_forecaster
Model version    : 1


In [0]:
import mlflow

# Register the already-logged model into the workspace (non-UC) model registry
mlflow.set_registry_uri("databricks")

workspace_model_name = "temperature_forecaster"

registered = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=workspace_model_name,
)

print(f"Workspace registry name : {registered.name}")
print(f"Version                 : {registered.version}")
print(f"Status                  : {registered.status}")

Successfully registered model 'temperature_forecaster'.
2026/07/26 04:53:44 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: temperature_forecaster, version 1
Created version '1' of model 'temperature_forecaster'.


Workspace registry name : temperature_forecaster
Version                 : 1
Status                  : PENDING_REGISTRATION


In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput, ServedEntityInput,
    AiGatewayConfig, AiGatewayInferenceTableConfig, AiGatewayUsageTrackingConfig,
)

w = WorkspaceClient()
registered_model_name = "dbx.default.temperature_forecaster"
catalog, schema, name = registered_model_name.split(".")
endpoint_name  = name                        # temperature_forecaster
monitoring_schema = f"{schema}_monitoring"  # default_monitoring

# Ensure the monitoring schema exists before endpoint creation
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{monitoring_schema}")

w.serving_endpoints.create(
    name=endpoint_name,
    config=EndpointCoreConfigInput(
        served_entities=[ServedEntityInput(
            entity_name=registered_model_name,
            entity_version="1",
            scale_to_zero_enabled=True,
            workload_size="Small",
        )]
    ),
    ai_gateway=AiGatewayConfig(
        inference_table_config=AiGatewayInferenceTableConfig(
            catalog_name=catalog,
            schema_name=monitoring_schema,
            table_name_prefix=endpoint_name,
            enabled=True,
        ),
        usage_tracking_config=AiGatewayUsageTrackingConfig(enabled=True),
    ),
)
print(f"Endpoint '{endpoint_name}' creation submitted — waiting for READY state in the next cell.")

Endpoint 'temperature_forecaster' creation submitted — waiting for READY state in the next cell.


In [0]:
import time
from databricks.sdk.service.serving import EndpointStateReady, EndpointStateConfigUpdate

def wait_for_endpoint_ready(name, timeout_s=900, poll_s=15):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        s = w.serving_endpoints.get(name).state
        if (
            s.ready == EndpointStateReady.READY
            and s.config_update == EndpointStateConfigUpdate.NOT_UPDATING
        ):
            print(f"Endpoint '{name}' is READY.")
            return
        print(f"  {s.ready} / {s.config_update} — retrying in {poll_s}s …")
        time.sleep(poll_s)
    raise TimeoutError(f"'{name}' not ready after {timeout_s}s")

wait_for_endpoint_ready(endpoint_name)

  EndpointStateReady.NOT_READY / EndpointStateConfigUpdate.IN_PROGRESS — retrying in 15s …
  EndpointStateReady.NOT_READY / EndpointStateConfigUpdate.IN_PROGRESS — retrying in 15s …
  EndpointStateReady.NOT_READY / EndpointStateConfigUpdate.IN_PROGRESS — retrying in 15s …
  EndpointStateReady.NOT_READY / EndpointStateConfigUpdate.IN_PROGRESS — retrying in 15s …
  EndpointStateReady.NOT_READY / EndpointStateConfigUpdate.IN_PROGRESS — retrying in 15s …
  EndpointStateReady.NOT_READY / EndpointStateConfigUpdate.IN_PROGRESS — retrying in 15s …
  EndpointStateReady.NOT_READY / EndpointStateConfigUpdate.IN_PROGRESS — retrying in 15s …
  EndpointStateReady.NOT_READY / EndpointStateConfigUpdate.IN_PROGRESS — retrying in 15s …
  EndpointStateReady.NOT_READY / EndpointStateConfigUpdate.IN_PROGRESS — retrying in 15s …
  EndpointStateReady.NOT_READY / EndpointStateConfigUpdate.IN_PROGRESS — retrying in 15s …
  EndpointStateReady.NOT_READY / EndpointStateConfigUpdate.IN_PROGRESS — retrying in 15s …